# 06 · LIBERO Shared-Interface Comparison: Liquid vs Pi0-Style

**Notebook version: 1.0.8**

Goal: keep data schema, encoders, training loop, and metrics identical across both policy heads.

This notebook is a methodology extension beyond the core three-task paper pipeline. It reuses the same fairness principle (shared encoders/backbone, head-only changes) in a LIBERO-style setup.

This version stages LIBERO HDF5 files to local Colab storage for training stability while still saving checkpoints/artifacts to persistent storage.

The current config is tuned toward a stronger Pi0 capacity for action prediction while keeping Liquid near half-size and using a larger batch on A100.

Use `01`–`04` notebooks for the canonical paper numbers and figures.

In [ ]:
# Notebook fingerprint (cache-bust verification)
from pathlib import Path
import subprocess

NOTEBOOK_VERSION = '1.0.8'
try:
    NOTEBOOK_GIT_COMMIT = subprocess.check_output(
        ['git', 'rev-parse', 'HEAD'],
        cwd=Path.cwd(),
        text=True,
    ).strip()
except Exception:
    NOTEBOOK_GIT_COMMIT = 'git-unavailable'

print(f'NOTEBOOK_VERSION={NOTEBOOK_VERSION}')
print(f'NOTEBOOK_GIT_COMMIT={NOTEBOOK_GIT_COMMIT[:12]}')

In [ ]:
import importlib.util
import os
from pathlib import Path
import subprocess
import sys

import yaml

IS_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ

# Keep LIBERO config local to the notebook runtime so imports are non-interactive.
LIBERO_CONFIG_HOME = Path('/content/.libero' if IS_COLAB else (Path.cwd() / '.libero'))
os.environ['LIBERO_CONFIG_PATH'] = str(LIBERO_CONFIG_HOME)

# Notebook-local dataset/cache root used to preseed LIBERO's config.
LIBERO_DATA_ROOT = Path('/content/libero_data' if IS_COLAB else (Path.cwd() / 'libero_data'))


def _pip(*pkgs):
    subprocess.check_call(
        [sys.executable, '-m', 'pip', '-q', '--disable-pip-version-check', 'install', *pkgs]
    )


def _preconfigure_libero():
    spec = importlib.util.find_spec('libero.libero')
    if spec is None or spec.origin is None:
        spec = importlib.util.find_spec('libero')
    if spec is None or spec.origin is None:
        raise RuntimeError('LIBERO is not installed, so its config cannot be preseeded.')

    package_root = Path(spec.origin).resolve().parent
    nested_root = package_root / 'libero'
    benchmark_root = nested_root if nested_root.exists() else package_root

    LIBERO_CONFIG_HOME.mkdir(parents=True, exist_ok=True)
    datasets_dir = LIBERO_DATA_ROOT / 'datasets'
    datasets_dir.mkdir(parents=True, exist_ok=True)

    config = {
        'benchmark_root': str(benchmark_root),
        'bddl_files': str(benchmark_root / 'bddl_files'),
        'init_states': str(benchmark_root / 'init_files'),
        'datasets': str(datasets_dir),
        'assets': str(benchmark_root / 'assets'),
    }

    config_path = LIBERO_CONFIG_HOME / 'config.yaml'
    with config_path.open('w') as f:
        yaml.safe_dump(config, f, sort_keys=False)

    print(f'LIBERO config preseeded at {config_path}')
    print(f"LIBERO datasets path: {config['datasets']}")


if IS_COLAB:
    _pip('torch', 'torchvision', 'torchaudio',
         '--index-url', 'https://download.pytorch.org/whl/cu124')
    _pip('transformers', 'accelerate', 'einops', 'tqdm', 'pyyaml', 'ncps',
         'wandb', 'huggingface_hub', 'h5py', 'ftfy', 'regex', 'sentencepiece',
         'imageio', 'imageio-ffmpeg')
    # LIBERO simulator — required for closed-loop rollout evaluation.
    # Needs MuJoCo; on most A100 Colab instances this installs cleanly.
    _pip('libero', 'robomimic', 'gymnasium')
    _preconfigure_libero()
    print('LIBERO simulator installed.')
else:
    # Local dev: install only packages that may be missing
    _pip('transformers', 'accelerate', 'huggingface_hub', 'h5py', 'ncps', 'wandb',
         'tqdm', 'einops', 'ftfy', 'regex', 'sentencepiece', 'imageio', 'imageio-ffmpeg')
    if importlib.util.find_spec('libero') is not None:
        _preconfigure_libero()

print(f'Install complete. IS_COLAB={IS_COLAB}')

In [ ]:
# ================================================================
#  TEST MODE
#
#  TEST = True   →  one full end-to-end pass:
#                   model download, data download, 1 training epoch,
#                   checkpoint save, and W&B upload.
#                   Takes ~5-10 min on A100, ~2 min on CPU (synthetic).
#
#  TEST = False  →  full production run using cfg.epochs / full dataset.
# ================================================================
TEST = False

In [ ]:
from dataclasses import dataclass
from datetime import datetime, timezone
import json
import math
import time
from typing import Dict, List

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

# IS_COLAB is set by the install cell; define fallback in case that cell was skipped
if 'IS_COLAB' not in globals():
    IS_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ

try:
    from transformers import CLIPModel, CLIPTokenizer
    HAS_TRANSFORMERS = True
except Exception:
    HAS_TRANSFORMERS = False

try:
    from ncps.torch import CfC
    HAS_NCPS = True
except Exception:
    HAS_NCPS = False

try:
    import wandb
    HAS_WANDB = True
except Exception:
    HAS_WANDB = False

try:
    import h5py
    HAS_H5PY = True
except ImportError:
    HAS_H5PY = False

try:
    from huggingface_hub import snapshot_download, hf_hub_download, list_repo_files
    HAS_HF_HUB = True
except ImportError:
    HAS_HF_HUB = False

LIBERO_IMPORT_STYLE = None
LIBERO_IMPORT_ERROR = None
try:
    import libero  # noqa: F401
    try:
        from libero.envs import OffScreenRenderEnv  # noqa: F401
        LIBERO_IMPORT_STYLE = 'libero.envs'
    except ImportError:
        from libero.libero.envs import OffScreenRenderEnv  # noqa: F401
        LIBERO_IMPORT_STYLE = 'libero.libero.envs'
    HAS_LIBERO = True
except ImportError as e:
    HAS_LIBERO = False
    LIBERO_IMPORT_ERROR = str(e)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(
    f'device={device} | ncps={HAS_NCPS} | transformers={HAS_TRANSFORMERS}'
    f' | wandb={HAS_WANDB} | h5py={HAS_H5PY} | hf_hub={HAS_HF_HUB}'
    f' | libero={HAS_LIBERO}'
)
if LIBERO_IMPORT_STYLE is not None:
    print(f'LIBERO import path: {LIBERO_IMPORT_STYLE}')
elif LIBERO_IMPORT_ERROR is not None:
    print(f'LIBERO import error: {LIBERO_IMPORT_ERROR}')

# --- JEPA-aligned MDN utilities ---
def mdn_nll_loss(logits, mu, log_sigma, target):
    # logits: [B,T,K], mu/log_sigma: [B,T,K,A], target: [B,T,A]
    target = target.unsqueeze(2)  # [B,T,1,A]
    z = (target - mu) * torch.exp(-log_sigma)
    log_prob = -0.5 * (z * z + 2.0 * log_sigma + math.log(2.0 * math.pi)).sum(dim=-1)  # [B,T,K]
    log_mix = F.log_softmax(logits, dim=-1) + log_prob
    nll = -torch.logsumexp(log_mix, dim=-1)  # [B,T]
    return nll.mean()

@torch.no_grad()
def sample_mdn_actions(logits, mu, log_sigma, K=10):
    # Returns [B,K,T,A]
    B, T, M = logits.shape
    A = mu.shape[-1]
    probs = F.softmax(logits, dim=-1)
    comp = torch.distributions.Categorical(probs=probs)
    idx = comp.sample((K,)).permute(1, 0, 2)  # [B,K,T]
    eps = torch.randn(B, K, T, A, device=mu.device)

    idx_exp = idx.unsqueeze(-1).unsqueeze(-1).expand(-1, -1, -1, 1, A)
    mu_k = mu.unsqueeze(1).expand(B, K, T, M, A).gather(3, idx_exp).squeeze(3)
    ls_k = log_sigma.unsqueeze(1).expand(B, K, T, M, A).gather(3, idx_exp).squeeze(3)
    return mu_k + torch.exp(ls_k) * eps

@torch.no_grad()
def mdn_mean_actions(logits, mu):
    w = F.softmax(logits, dim=-1)
    return (w.unsqueeze(-1) * mu).sum(dim=2)  # [B,T,A]

@torch.no_grad()
def compute_jerk(trajs):
    if trajs.shape[-2] < 3:
        return torch.zeros(trajs.shape[:-2], device=trajs.device)
    jerk = trajs[..., 2:, :] - 2.0 * trajs[..., 1:-1, :] + trajs[..., :-2, :]
    return (jerk ** 2).mean(dim=(-1, -2))

@torch.no_grad()
def compute_sample_metrics(samples, target):
    # samples: [B,K,T,A], target: [B,T,A]
    per_step_mse = ((samples - target.unsqueeze(1)) ** 2).mean(dim=-1)  # [B,K,T]
    traj_mse = per_step_mse.mean(dim=-1)  # [B,K]
    best_idx = traj_mse.argmin(dim=1)
    batch_idx = torch.arange(samples.shape[0], device=samples.device)
    best_step_mse = per_step_mse[batch_idx, best_idx]

    if samples.shape[1] > 1:
        diffs = samples.unsqueeze(2) - samples.unsqueeze(1)
        pairwise_rms = torch.sqrt((diffs ** 2).mean(dim=(-1, -2)) + 1e-12)
        tri = torch.triu_indices(samples.shape[1], samples.shape[1], offset=1, device=samples.device)
        diversity = pairwise_rms[:, tri[0], tri[1]].mean(dim=1).mean().item()
    else:
        diversity = 0.0

    return {
        'sample_mean_mse': traj_mse.mean().item(),
        'best_of_k_mse': traj_mse.min(dim=1).values.mean().item(),
        'diversity_l2': diversity,
        'smoothness': compute_jerk(samples).mean().item(),
        'per_horizon_best_of_k_mse': best_step_mse.mean(dim=0).detach().cpu().tolist(),
    }

device = cpu | ncps = False | transformers = True


In [ ]:
@dataclass
class Config:
    # Shared task setup
    obs_horizon: int = 2
    pred_horizon: int = 16
    state_dim: int = 14
    action_dim: int = 7
    vocab_size: int = 49408  # CLIP vocab size
    text_len: int = 32

    # Shared backbone (Pi0 reference)
    d_model: int = 768
    n_heads: int = 12
    n_layers: int = 10
    dropout: float = 0.1

    # Head sizes + size policy
    # Pi0 is intentionally scaled up here to favor best action-prediction quality.
    # Liquid is kept near half-size overall while remaining meaningfully competitive.
    pi0_hidden: int = 2304
    liquid_hidden: int = 1536
    num_mixtures: int = 5
    enforce_liquid_half_size: bool = True
    target_liquid_to_pi0_ratio: float = 0.50
    liquid_d_model: int = 768
    liquid_n_heads: int = 12
    liquid_n_layers: int = 6

    # Optimization
    lr: float = 1e-4
    wd: float = 1e-4
    epochs: int = 60
    batch_size: int = 128
    loader_num_workers: int = 4
    min_train_batches: int = 100

    # Encoder strategy
    use_pretrained_encoders: bool = True
    pretrained_name: str = 'openai/clip-vit-base-patch32'
    freeze_encoders: bool = True
    unfreeze_clip_vision_last_n: int = 0
    unfreeze_clip_text_last_n: int = 0
    unfreeze_clip_projections: bool = False
    image_size: int = 224

    # Two-branch liquid objective
    tf_ratio_start: float = 1.0
    tf_ratio_end: float = 0.15
    free_w_start: float = 0.20
    free_w_end: float = 0.75

    # Eval protocol
    k_values: List[int] = None

    # Closed-loop evaluation
    n_eval_episodes: int = 10
    eval_horizon: int = 300
    eval_k_samples: int = 10
    closed_loop_eval_tasks: int = 10

    # Dataset
    n_train: int = 512       # synthetic fallback size; ignored for real data
    n_val: int = 128
    libero_suite: str = 'LIBERO_90'
    libero_n_tasks: int = 90
    require_real_libero_data: bool = True
    min_libero_hdf5_files: int = 40
    stage_libero_to_local_on_colab: bool = True
    local_libero_root: str = '/content/libero_data'
    hdf5_read_retries: int = 3
    hdf5_retry_sleep_sec: float = 1.0

    # Storage + restart
    use_drive: bool = True
    drive_mount_point: str = '/content/drive'
    drive_root: str = '/content/drive/MyDrive/liquidnets_runs'
    local_root: str = '/content/liquidnets_runs'
    run_name: str = 'libero90_liquid_vs_pi0_scaled'
    resume: bool = False
    save_every_epochs: int = 5

    # Tracking
    enable_wandb: bool = True
    wandb_project: str = 'liquidnets-libero'
    wandb_entity: str = ''

cfg = Config(k_values=[1, 2, 5, 10])

# Apply TEST-mode overrides (TEST flag defined in cell above)
_test = globals().get('TEST', False)
if _test:
    cfg.epochs = 1
    cfg.n_train = 64
    cfg.n_val = 32
    cfg.batch_size = 8
    cfg.loader_num_workers = 0
    cfg.min_train_batches = 1
    cfg.libero_n_tasks = 1
    cfg.resume = False
    cfg.save_every_epochs = 1
    cfg.run_name = 'libero_liquid_vs_pi0_TEST'
    cfg.n_eval_episodes = 1
    cfg.closed_loop_eval_tasks = 1
    cfg.require_real_libero_data = False

NUM_EVAL_SAMPLES = max(cfg.k_values)
print(f'TEST={_test} | epochs={cfg.epochs} | batch_size={cfg.batch_size}'
      f' | workers={cfg.loader_num_workers} | libero_suite={cfg.libero_suite}'
      f' | n_tasks={cfg.libero_n_tasks} | n_eval_episodes={cfg.n_eval_episodes}')
print(f'stage_libero_to_local_on_colab={cfg.stage_libero_to_local_on_colab}'
      f' | local_libero_root={cfg.local_libero_root}')

In [ ]:
# Runtime paths, Drive mount (Colab), and optional W&B init
IS_COLAB = 'COLAB_GPU' in os.environ

if cfg.use_drive and IS_COLAB:
    try:
        from google.colab import drive  # type: ignore
        drive.mount(cfg.drive_mount_point, force_remount=False)
    except Exception as e:
        print(f'[WARN] Drive mount failed: {e}. Falling back to local path.')

root_path = Path(cfg.drive_root if (cfg.use_drive and Path(cfg.drive_mount_point).exists()) else cfg.local_root)
run_path = root_path / cfg.run_name
ckpt_path = run_path / 'checkpoints'
artifact_path = run_path / 'artifacts'

ckpt_path.mkdir(parents=True, exist_ok=True)
artifact_path.mkdir(parents=True, exist_ok=True)

latest_ckpt = ckpt_path / 'latest.pt'
best_liq_ckpt = ckpt_path / 'best_liquid.pt'
best_pi0_ckpt = ckpt_path / 'best_pi0.pt'
history_json = artifact_path / 'history.json'
results_json = artifact_path / 'results.json'

print('Run directory:', run_path)
print('Latest checkpoint:', latest_ckpt)

# Colab secret support: read WANDB_API_KEY from Secrets if available
if cfg.enable_wandb and IS_COLAB and 'WANDB_API_KEY' not in os.environ:
    try:
        from google.colab import userdata  # type: ignore
        _wb_key = userdata.get('WANDB_API_KEY')
        if _wb_key:
            os.environ['WANDB_API_KEY'] = _wb_key
            print('Loaded WANDB_API_KEY from Colab Secrets.')
        else:
            print('[INFO] Colab Secret WANDB_API_KEY is empty or not set.')
    except Exception as e:
        print(f'[INFO] Could not read Colab Secrets for W&B key: {e}')

wandb_run = None
if cfg.enable_wandb and HAS_WANDB:
    wb_kwargs = {
        'project': cfg.wandb_project,
        'name': cfg.run_name,
        'config': cfg.__dict__,
        'reinit': True,
    }
    if cfg.wandb_entity:
        wb_kwargs['entity'] = cfg.wandb_entity
    wandb_run = wandb.init(**wb_kwargs)
    print('W&B run initialized:', wandb_run.name)
elif cfg.enable_wandb:
    print('[WARN] wandb not installed in kernel; continuing without online logging.')

In [ ]:
# ---------------------------------------------------------------
# Optional: redirect HuggingFace cache to Drive so models survive
# Colab session restarts without re-downloading.
# ---------------------------------------------------------------
if cfg.use_drive and IS_COLAB and Path(cfg.drive_mount_point).exists():
    hf_cache = root_path / 'hf_cache'
    hf_cache.mkdir(parents=True, exist_ok=True)
    os.environ['HF_HOME'] = str(hf_cache)
    print(f'HF_HOME → {hf_cache}')

# ---------------------------------------------------------------
# CLIP model pre-download
# ---------------------------------------------------------------
clip_tokenizer = None
if cfg.use_pretrained_encoders:
    if HAS_HF_HUB:
        print(f'Pre-caching {cfg.pretrained_name} ...')
        snapshot_download(
            repo_id=cfg.pretrained_name,
            repo_type='model',
            ignore_patterns=['*.msgpack', '*.h5', 'flax_*', 'tf_*', 'rust_model.ot'],
        )
        print('  model weights cached.')
    else:
        print('[INFO] huggingface_hub not available; transformers will auto-download on first use.')

    if HAS_TRANSFORMERS:
        try:
            clip_tokenizer = CLIPTokenizer.from_pretrained(cfg.pretrained_name)
            print(f'  CLIP tokenizer loaded ({cfg.pretrained_name}).')
        except Exception as e:
            print(f'  [WARN] Tokenizer load failed: {e}. Task descriptions will use hash-based token fallback.')

# ---------------------------------------------------------------
# LIBERO dataset download / staging
#
# Persistent source lives on Drive when enabled; runtime training
# on Colab uses local SSD staging for HDF5 stability.
# ---------------------------------------------------------------
import shutil

LIBERO_HF_REPO = 'yifengzhu-hf/LIBERO-datasets'
LIBERO_SUITE_ALIASES = {
    'LIBERO_SPATIAL': 'libero_spatial',
    'LIBERO_OBJECT': 'libero_object',
    'LIBERO_GOAL': 'libero_goal',
    'LIBERO_10': 'libero_10',
    'LIBERO_90': 'libero_90',
    'LIBERO_LONG': 'libero_90',
}

def get_libero_suite_dir_name(suite_name: str) -> str:
    suite_name = str(suite_name).strip()
    return LIBERO_SUITE_ALIASES.get(suite_name, suite_name.lower())

def _copy_tree_if_needed(src_dir: Path, dst_dir: Path, suffix='*.hdf5'):
    dst_dir.mkdir(parents=True, exist_ok=True)
    copied = 0
    for src in sorted(src_dir.glob(suffix)):
        dst = dst_dir / src.name
        needs_copy = (not dst.exists()) or (src.stat().st_size != dst.stat().st_size)
        if needs_copy:
            shutil.copy2(src, dst)
            copied += 1
    return copied

LIBERO_PERSISTENT_PATH = root_path / 'libero_data'
LIBERO_PERSISTENT_PATH.mkdir(parents=True, exist_ok=True)

_libero_runtime_base = Path(cfg.local_libero_root) if (IS_COLAB and cfg.stage_libero_to_local_on_colab) else LIBERO_PERSISTENT_PATH
_libero_runtime_base.mkdir(parents=True, exist_ok=True)
LIBERO_LOCAL_PATH = _libero_runtime_base

libero_suite_dir = get_libero_suite_dir_name(cfg.libero_suite)
cfg.libero_suite = libero_suite_dir

libero_available = False
suite_dir_persistent = LIBERO_PERSISTENT_PATH / libero_suite_dir
suite_dir_runtime = LIBERO_LOCAL_PATH / libero_suite_dir

if HAS_H5PY and HAS_HF_HUB:
    existing_persistent = list(suite_dir_persistent.glob('*.hdf5')) if suite_dir_persistent.exists() else []
    if existing_persistent:
        print(f'LIBERO data already present in persistent store ({len(existing_persistent)} file(s)): {suite_dir_persistent}')
        libero_available = True
    else:
        try:
            print(f'Listing {LIBERO_HF_REPO} for suite {libero_suite_dir} ...')
            all_files = sorted([
                f for f in list_repo_files(LIBERO_HF_REPO, repo_type='dataset')
                if f.startswith(f'{libero_suite_dir}/') and f.endswith('.hdf5')
            ])[:cfg.libero_n_tasks]

            if all_files:
                print(f'Downloading {len(all_files)} task file(s) to persistent store:')
                for fname in all_files:
                    print(f'  {fname}')
                    hf_hub_download(
                        repo_id=LIBERO_HF_REPO,
                        filename=fname,
                        repo_type='dataset',
                        local_dir=str(LIBERO_PERSISTENT_PATH),
                    )
                print('Download complete.')
                libero_available = bool(list(suite_dir_persistent.glob('*.hdf5')))
            else:
                print(f'[WARN] No .hdf5 files found for {libero_suite_dir} in {LIBERO_HF_REPO}.')
                print('       Verify LIBERO_HF_REPO or download manually to:', suite_dir_persistent)
        except Exception as e:
            print(f'[WARN] LIBERO download failed: {e}')
            print('       Falling back to synthetic data for training.')
else:
    missing = [p for p, ok in [('h5py', HAS_H5PY), ('huggingface_hub', HAS_HF_HUB)] if not ok]
    print(f'[INFO] Missing: {missing}. Using synthetic data (install to enable real data).')

if suite_dir_persistent.exists() and suite_dir_persistent != suite_dir_runtime:
    print(f'Staging LIBERO suite to local runtime: {suite_dir_persistent} -> {suite_dir_runtime}')
    _copied = _copy_tree_if_needed(suite_dir_persistent, suite_dir_runtime, '*.hdf5')
    print(f'  staged/updated {_copied} file(s) on local runtime storage')
elif suite_dir_persistent.exists():
    suite_dir_runtime.mkdir(parents=True, exist_ok=True)

print(f'\nlibero_available={libero_available}'
      f' | clip_tokenizer={"loaded" if clip_tokenizer else "fallback hash"}')
print(f'LIBERO persistent path: {LIBERO_PERSISTENT_PATH}')
print(f'LIBERO runtime path   : {LIBERO_LOCAL_PATH}')

_suite_files = sorted(suite_dir_runtime.glob('*.hdf5')) if suite_dir_runtime.exists() else []
if getattr(cfg, 'require_real_libero_data', False):
    _min_files = int(getattr(cfg, 'min_libero_hdf5_files', 1))
    if len(_suite_files) < _min_files:
        raise RuntimeError(
            f'Real LIBERO data required but only found {len(_suite_files)} .hdf5 file(s) in {suite_dir_runtime}. '
            f'Need at least {_min_files}. Disable cfg.require_real_libero_data only for smoke tests.'
        )

In [ ]:
class Batch(dict):
    """Keys used by both models: image,state,text_ids,text_mask,action"""


class TinyVision(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, 2, 1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, 2, 1), nn.ReLU(),
            nn.Conv2d(64, 128, 3, 2, 1), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.proj = nn.Linear(128, d)

    def forward(self, x):
        B, T = x.shape[:2]
        h = self.net(x.reshape(B * T, *x.shape[2:])).flatten(1)
        return self.proj(h).view(B, T, -1)


class TinyText(nn.Module):
    def __init__(self, vocab_size, d):
        super().__init__()
        self.tok = nn.Embedding(vocab_size, d)

    def forward(self, text_ids, text_mask):
        h = self.tok(text_ids)
        m = text_mask.float().unsqueeze(-1)
        return (h * m).sum(1) / m.sum(1).clamp_min(1.0)


class PretrainedClipEncoders(nn.Module):
    """Pretrained vision+text interface that can be recycled across Liquid/Pi0 heads."""
    def __init__(self, cfg):
        super().__init__()
        self.enabled = bool(cfg.use_pretrained_encoders and HAS_TRANSFORMERS)
        self.image_size = int(cfg.image_size)
        self.vision_out_dim = cfg.d_model
        self.text_out_dim = cfg.d_model

        if self.enabled:
            try:
                clip = CLIPModel.from_pretrained(cfg.pretrained_name)
                self.vision = clip.vision_model
                self.text = clip.text_model
                self.visual_projection = clip.visual_projection
                self.text_projection = clip.text_projection
                self.vision_out_dim = int(self.visual_projection.out_features)
                self.text_out_dim = int(self.text_projection.out_features)
            except Exception as e:
                print(f'[WARN] Failed to load pretrained encoders: {e}. Falling back to tiny encoders.')
                self.enabled = False

        if not self.enabled:
            self.vision = TinyVision(cfg.d_model)
            self.text = TinyText(cfg.vocab_size, cfg.d_model)
            self.visual_projection = nn.Identity()
            self.text_projection = nn.Identity()

    def _preprocess_image(self, x):
        # x: [B,T,3,H,W], expected ~[0,1]
        B, T = x.shape[:2]
        x = x.reshape(B * T, *x.shape[2:])
        x = F.interpolate(x, size=(self.image_size, self.image_size), mode='bilinear', align_corners=False)
        # CLIP normalization
        mean = torch.tensor([0.48145466, 0.4578275, 0.40821073], device=x.device).view(1, 3, 1, 1)
        std = torch.tensor([0.26862954, 0.26130258, 0.27577711], device=x.device).view(1, 3, 1, 1)
        x = (x - mean) / std
        return x, B, T

    def encode_vision(self, image):
        if self.enabled:
            x, B, T = self._preprocess_image(image)
            out = self.vision(pixel_values=x)
            cls = out.last_hidden_state[:, 0]
            feat = self.visual_projection(cls)
            return feat.view(B, T, -1)
        return self.vision(image)

    def encode_text(self, text_ids, text_mask):
        if self.enabled:
            out = self.text(input_ids=text_ids, attention_mask=text_mask)
            cls = out.last_hidden_state[:, 0]
            return self.text_projection(cls)
        return self.text(text_ids, text_mask)


class SharedBackbone(nn.Module):
    def __init__(self, cfg, shared_encoders=None):
        super().__init__()
        self.cfg = cfg
        self.enc = shared_encoders if shared_encoders is not None else PretrainedClipEncoders(cfg)

        if cfg.freeze_encoders:
            for p in self.enc.parameters():
                p.requires_grad = False

            # Optional partial unfreeze for CLIP encoders
            v_last = int(getattr(cfg, 'unfreeze_clip_vision_last_n', 0))
            t_last = int(getattr(cfg, 'unfreeze_clip_text_last_n', 0))
            unfreeze_proj = bool(getattr(cfg, 'unfreeze_clip_projections', True))

            if getattr(self.enc, 'enabled', False):
                if v_last > 0 and hasattr(self.enc.vision, 'encoder') and hasattr(self.enc.vision.encoder, 'layers'):
                    v_layers = self.enc.vision.encoder.layers
                    for lyr in list(v_layers)[-v_last:]:
                        for p in lyr.parameters():
                            p.requires_grad = True

                if t_last > 0 and hasattr(self.enc.text, 'encoder') and hasattr(self.enc.text.encoder, 'layers'):
                    t_layers = self.enc.text.encoder.layers
                    for lyr in list(t_layers)[-t_last:]:
                        for p in lyr.parameters():
                            p.requires_grad = True

                if unfreeze_proj:
                    for p in self.enc.visual_projection.parameters():
                        p.requires_grad = True
                    for p in self.enc.text_projection.parameters():
                        p.requires_grad = True

            if not any(p.requires_grad for p in self.enc.parameters()):
                self.enc.eval()
            else:
                _n_trainable = sum(int(p.numel()) for p in self.enc.parameters() if p.requires_grad)
                print(f'[INFO] Partial encoder unfreeze active: trainable encoder params={_n_trainable:,}')

        self.vis_proj = nn.Linear(self.enc.vision_out_dim, cfg.d_model)
        self.state_proj = nn.Linear(cfg.state_dim, cfg.d_model)
        self.text_proj = nn.Linear(self.enc.text_out_dim, cfg.d_model)

        layer = nn.TransformerEncoderLayer(
            cfg.d_model,
            cfg.n_heads,
            4 * cfg.d_model,
            cfg.dropout,
            batch_first=True,
        )
        self.temporal = nn.TransformerEncoder(layer, cfg.n_layers)

    def forward(self, batch):
        with torch.set_grad_enabled(any(p.requires_grad for p in self.enc.parameters())):
            v = self.enc.encode_vision(batch['image'])
            t = self.enc.encode_text(batch['text_ids'], batch['text_mask'])

        v = self.vis_proj(v)
        s = self.state_proj(batch['state'])
        t = self.text_proj(t).unsqueeze(1).expand(-1, v.size(1), -1)
        return self.temporal(v + s + t)

In [5]:
class MdnHead(nn.Module):
    def __init__(self, in_dim, action_dim, num_mixtures):
        super().__init__()
        self.action_dim = action_dim
        self.num_mixtures = num_mixtures
        out_dim = num_mixtures * (1 + 2 * action_dim)
        self.proj = nn.Linear(in_dim, out_dim)

    def forward(self, h):
        # h: [B,T,H]
        B, T, _ = h.shape
        raw = self.proj(h).view(B, T, self.num_mixtures, 1 + 2 * self.action_dim)
        logits = raw[..., 0]
        mu = raw[..., 1:1 + self.action_dim]
        log_sigma = raw[..., 1 + self.action_dim:]
        log_sigma = torch.clamp(log_sigma, min=-7.0, max=2.0)
        return logits, mu, log_sigma


class BasePolicy(nn.Module):
    def __init__(self, backbone, cfg):
        super().__init__()
        self.backbone = backbone
        self.cfg = cfg

    def predict_mdn(self, z, batch, action_target=None, teacher_forcing_ratio=0.0):
        raise NotImplementedError

    def forward(self, batch, action_target=None, teacher_forcing_ratio=0.0, return_mdn=True):
        z = self.backbone(batch)
        logits, mu, log_sigma = self.predict_mdn(
            z,
            batch,
            action_target=action_target,
            teacher_forcing_ratio=teacher_forcing_ratio,
        )
        pred_mean = mdn_mean_actions(logits, mu)
        out = {
            'pred_actions': pred_mean,
            'aux': {
                'logits': logits,
                'mu': mu,
                'log_sigma': log_sigma,
            }
        }
        return out

    def loss(self, out, batch):
        logits = out['aux']['logits']
        mu = out['aux']['mu']
        log_sigma = out['aux']['log_sigma']
        nll = mdn_nll_loss(logits, mu, log_sigma, batch['action'])
        mse = F.mse_loss(out['pred_actions'], batch['action'])
        return {'loss': nll, 'nll': nll.detach(), 'mse': mse.detach()}


class LiquidPolicy(BasePolicy):
    def __init__(self, backbone, cfg):
        super().__init__(backbone, cfg)
        if HAS_NCPS:
            self.core = CfC(cfg.d_model, cfg.liquid_hidden, batch_first=True)
        else:
            self.core = nn.GRU(cfg.d_model, cfg.liquid_hidden, num_layers=2, batch_first=True)

        self.action_embed = nn.Linear(cfg.action_dim, cfg.d_model)
        self.decoder = nn.GRUCell(cfg.d_model, cfg.liquid_hidden)
        self.mdn = MdnHead(cfg.liquid_hidden, cfg.action_dim, cfg.num_mixtures)

    def predict_mdn(self, z, batch, action_target=None, teacher_forcing_ratio=0.0):
        B, _, _ = z.shape
        T = self.cfg.pred_horizon

        h_enc, _ = self.core(z)
        h = h_enc[:, -1]  # [B,H]

        prev_action = torch.zeros(B, self.cfg.action_dim, device=z.device)
        hidden_seq = []

        for t in range(T):
            use_teacher = (
                (action_target is not None) and
                (torch.rand(1, device=z.device).item() < float(teacher_forcing_ratio))
            )
            inp_action = action_target[:, t] if use_teacher else prev_action
            inp = self.action_embed(inp_action)
            h = self.decoder(inp, h)
            hidden_seq.append(h.unsqueeze(1))

            # rollout with mean prediction
            logits_t, mu_t, _ = self.mdn(h.unsqueeze(1))
            prev_action = mdn_mean_actions(logits_t, mu_t).squeeze(1)

        hidden_seq = torch.cat(hidden_seq, dim=1)  # [B,T,H]
        return self.mdn(hidden_seq)


class Pi0StylePolicy(BasePolicy):
    def __init__(self, backbone, cfg):
        super().__init__(backbone, cfg)
        self.query = nn.Parameter(torch.randn(1, cfg.pred_horizon, cfg.d_model) * 0.02)
        layer = nn.TransformerDecoderLayer(
            d_model=cfg.d_model,
            nhead=cfg.n_heads,
            dim_feedforward=4 * cfg.d_model,
            dropout=cfg.dropout,
            batch_first=True,
        )
        self.dec = nn.TransformerDecoder(layer, cfg.n_layers)
        self.proj = nn.Linear(cfg.d_model, cfg.pi0_hidden)
        self.mdn = MdnHead(cfg.pi0_hidden, cfg.action_dim, cfg.num_mixtures)

    def predict_mdn(self, z, batch, action_target=None, teacher_forcing_ratio=0.0):
        q = self.query.expand(z.size(0), -1, -1)
        y = self.dec(q, z)
        h = self.proj(y)
        return self.mdn(h)

In [ ]:
class LiberoHDF5Dataset(Dataset):
    """Load LIBERO demonstrations from .hdf5 files.

    Expected HDF5 structure (standard LIBERO / robomimic format):
        data/
            demo_0/
                obs/
                    agentview_image:     (T, H, W, 3)  uint8
                    robot0_eef_pos:      (T, 3)
                    robot0_eef_quat:     (T, 4)
                    robot0_gripper_qpos: (T, 2)
                    robot0_joint_pos:    (T, 7)
                actions: (T, 7)
    """

    CACHE_VERSION = 1

    @staticmethod
    def _default_cache_root(cfg):
        if 'root_path' in globals():
            return Path(globals()['root_path'])

        local_root = Path(getattr(cfg, 'local_root', '/tmp/liquidnets_runs'))
        drive_root = Path(getattr(cfg, 'drive_root', '/tmp/liquidnets_runs'))
        drive_ok = bool(getattr(cfg, 'use_drive', False)) and \
                  Path(getattr(cfg, 'drive_mount_point', '/x')).exists()
        return drive_root if drive_ok else local_root

    @classmethod
    def _cache_file(cls, cfg, split):
        root = cls._default_cache_root(cfg)
        cache_dir = root / 'cache' / 'libero_hdf5_index'
        cache_dir.mkdir(parents=True, exist_ok=True)

        suite_name = str(getattr(cfg, 'libero_suite', 'libero')).lower()
        fname = (
            f'{suite_name}_{split}'
            f'_oh{int(getattr(cfg, "obs_horizon", 0))}'
            f'_ph{int(getattr(cfg, "pred_horizon", 0))}'
            '.json'
        )
        return cache_dir / fname

    @staticmethod
    def _file_signature(hdf5_paths):
        sig = []
        for p in sorted([Path(x) for x in hdf5_paths], key=lambda x: str(x)):
            st = p.stat()
            sig.append({
                'path': str(p.resolve()),
                'size': int(st.st_size),
                'mtime_ns': int(getattr(st, 'st_mtime_ns', int(st.st_mtime * 1e9))),
            })
        return sig

    @staticmethod
    def _cfg_signature(cfg):
        return {
            'obs_horizon': int(getattr(cfg, 'obs_horizon', 0)),
            'pred_horizon': int(getattr(cfg, 'pred_horizon', 0)),
        }

    @classmethod
    def _try_load_index_cache(cls, cache_file, split, cfg_sig, file_sig):
        if not cache_file.exists():
            return None
        try:
            with cache_file.open('r') as f:
                payload = json.load(f)
            if payload.get('version') != cls.CACHE_VERSION:
                return None
            if payload.get('split') != split:
                return None
            if payload.get('cfg') != cfg_sig:
                return None
            if payload.get('files') != file_sig:
                return None
            demos = payload.get('demos', None)
            if not isinstance(demos, list):
                return None
            return demos
        except Exception:
            return None

    @staticmethod
    def _write_index_cache(cache_file, split, cfg_sig, file_sig, demos):
        payload = {
            'version': LiberoHDF5Dataset.CACHE_VERSION,
            'split': split,
            'cfg': cfg_sig,
            'files': file_sig,
            'demos': demos,
            'saved_at_utc': datetime.now(timezone.utc).isoformat(),
        }
        tmp = cache_file.with_suffix('.tmp')
        with tmp.open('w') as f:
            json.dump(payload, f)
        tmp.replace(cache_file)

    def __init__(self, cfg, hdf5_paths, split='train', clip_tokenizer=None):
        self.cfg = cfg
        self.clip_tokenizer = clip_tokenizer
        self.demos = []

        hdf5_paths = [Path(p) for p in hdf5_paths]
        cache_file = self._cache_file(cfg, split)
        cfg_sig = self._cfg_signature(cfg)
        file_sig = self._file_signature(hdf5_paths)

        t0 = time.perf_counter()
        cached = self._try_load_index_cache(cache_file, split, cfg_sig, file_sig)
        if cached is not None:
            self.demos = cached
            dt = time.perf_counter() - t0
            print(f'LiberoHDF5Dataset ({split}): {len(self.demos)} windows '
                  f'from {len(hdf5_paths)} task file(s).')
            print(f'  [CACHE] loaded index from {cache_file} in {dt:.2f}s')
            return

        print(f'  [CACHE] rebuilding index for split="{split}" over {len(hdf5_paths)} file(s)...')
        _path_iter = tqdm(
            hdf5_paths,
            desc=f'Indexing LIBERO ({split})',
            unit='file',
            leave=False,
            dynamic_ncols=True,
        )
        for path in _path_iter:
            with h5py.File(str(path), 'r') as f:
                # Task name: try metadata first, fall back to filename
                task_name = path.stem.replace('_demo', '').replace('_', ' ')
                if 'problem_info' in f.attrs:
                    try:
                        task_name = json.loads(f.attrs['problem_info']).get('problem_name', task_name)
                    except Exception:
                        pass
                elif 'task_description' in f.attrs:
                    task_name = str(f.attrs['task_description'])

                demo_keys = sorted(
                    [k for k in f['data'].keys() if k.startswith('demo_')],
                    key=lambda x: int(x.split('_')[-1]),
                )
                n = len(demo_keys)
                split_idx = max(1, int(0.9 * n))
                keys = demo_keys[:split_idx] if split == 'train' else demo_keys[split_idx:]

                for k in keys:
                    T_full = f['data'][k]['actions'].shape[0]
                    if T_full < cfg.obs_horizon + cfg.pred_horizon:
                        continue
                    self.demos.append({
                        'path': str(path),
                        'key': k,
                        'T': T_full,
                        'task_name': task_name,
                    })

            if hasattr(_path_iter, 'set_postfix'):
                _path_iter.set_postfix(windows=len(self.demos), refresh=False)

        if hasattr(_path_iter, 'close'):
            _path_iter.close()

        dt = time.perf_counter() - t0
        print(f'LiberoHDF5Dataset ({split}): {len(self.demos)} windows '
              f'from {len(hdf5_paths)} task file(s).')
        print(f'  [CACHE] rebuilt index in {dt:.2f}s; saving to {cache_file}')
        try:
            self._write_index_cache(cache_file, split, cfg_sig, file_sig, self.demos)
        except Exception as e:
            print(f'  [WARN] failed to save index cache: {e}')

    def __len__(self):
        return len(self.demos)

    def _tokenize(self, text):
        if self.clip_tokenizer is not None:
            enc = self.clip_tokenizer(
                text, return_tensors='pt', padding='max_length',
                max_length=self.cfg.text_len, truncation=True,
            )
            return enc['input_ids'].squeeze(0), enc['attention_mask'].squeeze(0)
        # Fallback: deterministic hash into CLIP vocab
        ids = torch.zeros(self.cfg.text_len, dtype=torch.long)
        mask = torch.zeros(self.cfg.text_len, dtype=torch.long)
        for i, word in enumerate(text.lower().split()[:self.cfg.text_len]):
            ids[i] = abs(hash(word)) % self.cfg.vocab_size
            mask[i] = 1
        return ids, mask

    @staticmethod
    def _pick_first_existing_key(group, candidates):
        for k in candidates:
            if k in group:
                return k
        return None

    @staticmethod
    def _get_obs_slice(obs_group, candidates, obs_s, obs_e, out_dim, default=0.0):
        key = LiberoHDF5Dataset._pick_first_existing_key(obs_group, candidates)
        if key is None:
            return np.full((obs_e - obs_s, out_dim), default, dtype=np.float32)

        arr = np.asarray(obs_group[key][obs_s:obs_e], dtype=np.float32)
        if arr.ndim == 1:
            arr = arr[:, None]

        # Right-pad/truncate to expected dim for robustness across schema variants
        if arr.shape[-1] < out_dim:
            pad = np.full((arr.shape[0], out_dim - arr.shape[-1]), default, dtype=np.float32)
            arr = np.concatenate([arr, pad], axis=-1)
        elif arr.shape[-1] > out_dim:
            arr = arr[:, :out_dim]

        return arr

    @staticmethod
    def _get_image_slice(obs_group, obs_s, obs_e):
        # Prefer canonical LIBERO camera names first, then fall back to any image/rgb key
        preferred = [
            'agentview_image',
            'robot0_eye_in_hand_image',
            'frontview_image',
            'rgb',
            'image',
        ]
        key = LiberoHDF5Dataset._pick_first_existing_key(obs_group, preferred)
        if key is None:
            dynamic = [k for k in obs_group.keys() if ('image' in k.lower() or 'rgb' in k.lower())]
            key = dynamic[0] if dynamic else None

        if key is None:
            raise KeyError(
                f"No image observation key found. Available obs keys: {list(obs_group.keys())}"
            )

        imgs = np.asarray(obs_group[key][obs_s:obs_e])

        # Normalize to (T, H, W, 3)
        if imgs.ndim != 4:
            raise ValueError(
                f"Expected image tensor with 4 dims, got shape {imgs.shape} from key '{key}'"
            )

        # Handle TCHW -> THWC if needed
        if imgs.shape[1] in (1, 3, 4) and imgs.shape[-1] not in (1, 3, 4):
            imgs = np.transpose(imgs, (0, 2, 3, 1))

        if imgs.shape[-1] == 1:
            imgs = np.repeat(imgs, 3, axis=-1)
        elif imgs.shape[-1] >= 3:
            imgs = imgs[..., :3]
        else:
            raise ValueError(f"Unsupported image channel shape {imgs.shape} from key '{key}'")

        return imgs.astype(np.float32) / 255.0

    def __getitem__(self, idx):
        info = self.demos[idx % len(self.demos)]
        with h5py.File(info['path'], 'r') as f:
            demo = f['data'][info['key']]
            obs = demo['obs']

            T_full = demo['actions'].shape[0]
            max_start = T_full - self.cfg.obs_horizon - self.cfg.pred_horizon
            start = np.random.randint(0, max(1, max_start + 1))
            obs_s, obs_e = start, start + self.cfg.obs_horizon
            act_s, act_e = obs_e, obs_e + self.cfg.pred_horizon

            # Images: robust key selection; output (T,C,H,W), float32 [0, 1]
            imgs = self._get_image_slice(obs, obs_s, obs_e)
            imgs = torch.from_numpy(imgs).permute(0, 3, 1, 2)

            # State: eef_pos(3) + eef_quat(4) + gripper(2) + joint(5) = 14D
            eef_pos = self._get_obs_slice(obs, ['robot0_eef_pos', 'eef_pos'], obs_s, obs_e, out_dim=3)
            eef_quat = self._get_obs_slice(obs, ['robot0_eef_quat', 'eef_quat'], obs_s, obs_e, out_dim=4)
            gripper = self._get_obs_slice(obs, ['robot0_gripper_qpos', 'gripper_qpos'], obs_s, obs_e, out_dim=2)
            joint = self._get_obs_slice(obs, ['robot0_joint_pos', 'joint_pos'], obs_s, obs_e, out_dim=5)
            state = np.concatenate([eef_pos, eef_quat, gripper, joint], axis=-1)
            state = torch.from_numpy(state.astype(np.float32))

            action_key = 'actions' if 'actions' in demo else ('action' if 'action' in demo else None)
            if action_key is None:
                raise KeyError(f"No action key found. Available demo keys: {list(demo.keys())}")
            actions = torch.from_numpy(np.asarray(demo[action_key][act_s:act_e], dtype=np.float32))

        text_ids, text_mask = self._tokenize(info['task_name'])
        return Batch(image=imgs, state=state, text_ids=text_ids,
                     text_mask=text_mask, action=actions)


class LiberoAdapterDataset(Dataset):
    """Uses real LIBERO HDF5 data when available; falls back to synthetic tensors."""

    def __init__(self, cfg, n=None, split='train', libero_path=None, clip_tokenizer=None):
        self.cfg = cfg
        self._real_ds = None
        n_default = cfg.n_train if split == 'train' else cfg.n_val
        self.n = n if n is not None else n_default

        # Resolve path to the suite directory
        resolved = libero_path
        if resolved is None:
            local_root = Path(getattr(cfg, 'local_root', '/tmp/liquidnets_runs'))
            drive_root = Path(getattr(cfg, 'drive_root', '/tmp/liquidnets_runs'))
            drive_ok = getattr(cfg, 'use_drive', False) and \
                       Path(getattr(cfg, 'drive_mount_point', '/x')).exists()
            root = drive_root if drive_ok else local_root
            suite_name = str(getattr(cfg, 'libero_suite', 'libero_spatial')).lower()
            resolved = root.parent / 'libero_data' / suite_name

        if HAS_H5PY and Path(resolved).exists():
            hdf5_files = sorted(Path(resolved).glob('*.hdf5'))
            if hdf5_files:
                try:
                    ds = LiberoHDF5Dataset(cfg, hdf5_files, split=split,
                                          clip_tokenizer=clip_tokenizer)
                    if len(ds) > 0:
                        self._real_ds = ds
                        return
                except Exception as e:
                    print(f'[WARN] HDF5 load failed: {e}. Falling back to synthetic data.')

        print(f'LiberoAdapterDataset ({split}): synthetic, n={self.n}')

    def __len__(self):
        return len(self._real_ds) if self._real_ds is not None else self.n

    def __getitem__(self, idx):
        if self._real_ds is not None:
            return self._real_ds[idx % len(self._real_ds)]
        return Batch(
            image=torch.randn(self.cfg.obs_horizon, 3, 128, 128),
            state=torch.randn(self.cfg.obs_horizon, self.cfg.state_dim),
            text_ids=torch.randint(0, self.cfg.vocab_size, (self.cfg.text_len,)),
            text_mask=torch.ones(self.cfg.text_len, dtype=torch.long),
            action=torch.randn(self.cfg.pred_horizon, self.cfg.action_dim),
        )


def collate(items):
    out = Batch()
    for k in items[0].keys():
        out[k] = torch.stack([x[k] for x in items], 0)
    return out


def to_device(batch, d):
    return Batch({k: (v.to(d) if torch.is_tensor(v) else v) for k, v in batch.items()})


# ─────────────────────────────────────────────────────────────────────────────
# Closed-loop env factory + obs→Batch tokenizer (require HAS_LIBERO = True)
# ─────────────────────────────────────────────────────────────────────────────

def get_task_metadata_from_hdf5(hdf5_path):
    """Extract task_name + bddl_file_name from a LIBERO HDF5 file."""
    task_name = None
    bddl_file_name = None
    hdf5_path = Path(hdf5_path)

    if HAS_H5PY and hdf5_path.exists():
        try:
            with h5py.File(str(hdf5_path), 'r') as _hf:
                raw_info = _hf.attrs.get('problem_info', '{}')
                if isinstance(raw_info, (bytes, bytearray)):
                    raw_info = raw_info.decode('utf-8', errors='ignore')

                info = {}
                if isinstance(raw_info, str):
                    try:
                        info = json.loads(raw_info)
                    except Exception:
                        info = {}
                elif isinstance(raw_info, dict):
                    info = raw_info

                for k in ['problem_name', 'task_name', 'task', 'language_instruction']:
                    if info.get(k):
                        task_name = str(info[k])
                        break

                for k in ['bddl_file_name', 'bddl_file', 'bddl_filename', 'bddl']:
                    if info.get(k):
                        bddl_file_name = str(info[k])
                        break

                if bddl_file_name is None:
                    for k in ['bddl_file_name', 'bddl_file', 'bddl_filename', 'bddl']:
                        if k in _hf.attrs:
                            bddl_file_name = str(_hf.attrs[k])
                            break
        except Exception:
            pass

    if not task_name:
        task_name = hdf5_path.stem.replace('_demo', '').replace('_', ' ')

    return {'task_name': task_name, 'bddl_file_name': bddl_file_name}


def make_libero_env_fn(task_name: str, libero_path=None, image_size: int = 224, bddl_file_name: str = None):
    """Return a callable(seed) -> gym env for a single LIBERO task.

    The returned factory is passed to evaluate_closed_loop as make_env_fn.
    LIBERO's OffScreenRenderEnv wraps the task as a gymnasium-compatible env.
    """
    def _make(seed: int = 0):
        if not HAS_LIBERO:
            raise RuntimeError(
                'LIBERO simulator not installed. '
                'Re-run the install cell — libero/robomimic/gymnasium must install cleanly.'
            )

        # Support multiple LIBERO constructor signatures across versions.
        cam_kwargs = {
            'camera_heights': image_size,
            'camera_widths': image_size,
            'camera_names': 'agentview',
        }

        attempt_kwargs = []
        if task_name and bddl_file_name:
            attempt_kwargs.append({'task_name': task_name, 'bddl_file_name': bddl_file_name})
        if bddl_file_name:
            attempt_kwargs.append({'bddl_file_name': bddl_file_name})
        if task_name:
            attempt_kwargs.append({'task_name': task_name})

        env = None
        last_error = None
        for kw in attempt_kwargs:
            try:
                env = OffScreenRenderEnv(**kw, **cam_kwargs)
                break
            except TypeError as e:
                last_error = e

        if env is None:
            tried = ', '.join([str(x) for x in attempt_kwargs])
            raise RuntimeError(
                f'Failed to construct OffScreenRenderEnv. Tried: {tried}. Last error: {last_error}'
            )
        if hasattr(env, 'seed'):
            env.seed(seed)
        return env
    return _make


def make_tokenizer_fn(cfg, clip_tokenizer=None, task_name: str = ''):
    """Return a callable(obs_dict) -> Batch[B=1] for use in closed_loop_rollout.

    obs_dict is the raw dict returned by LIBERO's env.reset() / env.step().
    The _task_name key (injected by the wrapper or passed at construction)
    is used for text conditioning; falls back to empty tokens.
    """
    def _tokenize_obs(obs: dict) -> 'Batch':
        # ── Image ──────────────────────────────────────────────────────────
        # LIBERO returns 'agentview_image' as (H, W, 3) uint8
        img_np = obs.get('agentview_image', np.zeros((224, 224, 3), dtype=np.uint8))
        img = torch.from_numpy(img_np.astype(np.float32) / 255.0).permute(2, 0, 1)  # (C,H,W)
        # Duplicate the single frame across obs_horizon (receding-horizon = step-by-step)
        image = img.unsqueeze(0).expand(cfg.obs_horizon, -1, -1, -1).clone()  # (T,C,H,W)

        # ── Proprioception ─────────────────────────────────────────────────
        eef_pos  = np.asarray(obs.get('robot0_eef_pos',      [0.0]*3),  dtype=np.float32)[:3]
        eef_quat = np.asarray(obs.get('robot0_eef_quat',     [0.0]*4),  dtype=np.float32)[:4]
        gripper  = np.asarray(obs.get('robot0_gripper_qpos', [0.0]*2),  dtype=np.float32)[:2]
        joint    = np.asarray(obs.get('robot0_joint_pos',    [0.0]*7),  dtype=np.float32)[:5]
        state_np = np.concatenate([eef_pos, eef_quat, gripper, joint])  # (14,)
        state = torch.from_numpy(state_np).unsqueeze(0).expand(cfg.obs_horizon, -1).clone()

        # ── Language ───────────────────────────────────────────────────────
        _name = obs.get('_task_name', task_name)  # env wrapper may inject this
        if clip_tokenizer is not None and _name:
            enc = clip_tokenizer(
                _name, return_tensors='pt', padding='max_length',
                max_length=cfg.text_len, truncation=True,
            )
            text_ids  = enc['input_ids'].squeeze(0)
            text_mask = enc['attention_mask'].squeeze(0)
        else:
            text_ids  = torch.zeros(cfg.text_len, dtype=torch.long)
            text_mask = torch.zeros(cfg.text_len, dtype=torch.long)

        # ── Dummy action (schema requirement; unused during inference) ─────
        action = torch.zeros(cfg.pred_horizon, cfg.action_dim)

        return Batch(
            image=image.unsqueeze(0),       # (1,T,C,H,W)
            state=state.unsqueeze(0),       # (1,T,14)
            text_ids=text_ids.unsqueeze(0),    # (1,L)
            text_mask=text_mask.unsqueeze(0),  # (1,L)
            action=action.unsqueeze(0),     # (1,T,A)
        )
    return _tokenize_obs

print('Dataset helpers ready.'
      f'  HAS_H5PY={HAS_H5PY}  HAS_LIBERO={HAS_LIBERO}')


In [ ]:
# Colab/Drive robustness patch: retry transient HDF5 reads and prefer locally staged files
import errno
import time as _time

def _libero_hdf5_getitem_with_retry(self, idx):
    info = self.demos[idx % len(self.demos)]
    retries = max(1, int(getattr(self.cfg, 'hdf5_read_retries', 1)))
    sleep_sec = float(getattr(self.cfg, 'hdf5_retry_sleep_sec', 0.5))

    last_err = None
    for attempt in range(1, retries + 1):
        try:
            with h5py.File(info['path'], 'r') as f:
                demo = f['data'][info['key']]
                obs = demo['obs']

                T_full = demo['actions'].shape[0]
                max_start = T_full - self.cfg.obs_horizon - self.cfg.pred_horizon
                start = np.random.randint(0, max(1, max_start + 1))
                obs_s, obs_e = start, start + self.cfg.obs_horizon
                act_s, act_e = obs_e, obs_e + self.cfg.pred_horizon

                imgs = self._get_image_slice(obs, obs_s, obs_e)
                imgs = torch.from_numpy(imgs).permute(0, 3, 1, 2)

                eef_pos = self._get_obs_slice(obs, ['robot0_eef_pos', 'eef_pos'], obs_s, obs_e, out_dim=3)
                eef_quat = self._get_obs_slice(obs, ['robot0_eef_quat', 'eef_quat'], obs_s, obs_e, out_dim=4)
                gripper = self._get_obs_slice(obs, ['robot0_gripper_qpos', 'gripper_qpos'], obs_s, obs_e, out_dim=2)
                joint = self._get_obs_slice(obs, ['robot0_joint_pos', 'joint_pos'], obs_s, obs_e, out_dim=5)
                state = np.concatenate([eef_pos, eef_quat, gripper, joint], axis=-1)
                state = torch.from_numpy(state.astype(np.float32))

                action_key = 'actions' if 'actions' in demo else ('action' if 'action' in demo else None)
                if action_key is None:
                    raise KeyError(f"No action key found. Available demo keys: {list(demo.keys())}")
                actions = torch.from_numpy(np.asarray(demo[action_key][act_s:act_e], dtype=np.float32))

            text_ids, text_mask = self._tokenize(info['task_name'])
            return Batch(image=imgs, state=state, text_ids=text_ids,
                         text_mask=text_mask, action=actions)
        except OSError as e:
            last_err = e
            _is_io = getattr(e, 'errno', None) in (errno.EIO, 5, None)
            if attempt < retries and _is_io:
                print(f'[WARN] Transient HDF5 read failure on attempt {attempt}/{retries}: {info["path"]} :: {e}')
                _time.sleep(sleep_sec * attempt)
                continue
            raise
        except Exception as e:
            last_err = e
            raise

    raise last_err if last_err is not None else RuntimeError('Unknown HDF5 read failure')

LiberoHDF5Dataset.__getitem__ = _libero_hdf5_getitem_with_retry
print('Robust HDF5 retry patch active.')

In [ ]:
# Consolidated closed-loop metadata/env compatibility (integrated path)
def _infer_bddl_from_hdf5_path(hdf5_path):
    p = Path(hdf5_path)
    stem = p.stem
    if stem.endswith('_demo'):
        stem = stem[:-5]
    return f"{stem}.bddl"


def _get_libero_bddl_root():
    """Resolve LIBERO bddl_files root from config/env when available."""
    cfg_path = os.environ.get('LIBERO_CONFIG_PATH', '')
    if cfg_path:
        cfg_file = Path(cfg_path) / 'config.yaml'
        if cfg_file.exists():
            try:
                with cfg_file.open('r') as f:
                    cfg_data = yaml.safe_load(f) or {}
                root = cfg_data.get('bddl_files', None)
                if root:
                    root_p = Path(root)
                    if root_p.exists():
                        return root_p
            except Exception:
                pass

    try:
        spec = importlib.util.find_spec('libero.libero')
        if spec is None or spec.origin is None:
            spec = importlib.util.find_spec('libero')
        if spec and spec.origin:
            package_root = Path(spec.origin).resolve().parent
            nested_root = package_root / 'libero'
            benchmark_root = nested_root if nested_root.exists() else package_root
            bddl_root = benchmark_root / 'bddl_files'
            if bddl_root.exists():
                return bddl_root
    except Exception:
        pass

    return None


def _resolve_bddl_path(bddl_file_name, libero_path=None):
    """Return absolute bddl path if found; else return normalized filename."""
    if not bddl_file_name:
        return None

    bddl = str(bddl_file_name)
    if not bddl.endswith('.bddl'):
        bddl = f"{bddl}.bddl"

    bddl_p = Path(bddl)
    if bddl_p.is_absolute() and bddl_p.exists():
        return str(bddl_p)

    bddl_root = _get_libero_bddl_root()
    suite_name = Path(libero_path).name if libero_path is not None else None

    candidates = []
    if bddl_root is not None:
        candidates.extend([
            bddl_root / bddl,
            bddl_root / Path(bddl).name,
        ])
        if suite_name:
            candidates.extend([
                bddl_root / suite_name / bddl,
                bddl_root / suite_name / Path(bddl).name,
            ])

    for c in candidates:
        if c.exists():
            return str(c)

    if bddl_root is not None:
        target_name = Path(bddl).name
        matches = sorted(bddl_root.rglob(target_name))
        if matches:
            return str(matches[0])

    return bddl


def get_task_metadata_from_hdf5(hdf5_path):
    """Extract task_name + bddl_file_name from a LIBERO HDF5 file (with filename fallback)."""
    task_name = None
    bddl_file_name = None
    hdf5_path = Path(hdf5_path)

    if HAS_H5PY and hdf5_path.exists():
        try:
            with h5py.File(str(hdf5_path), 'r') as _hf:
                raw_info = _hf.attrs.get('problem_info', '{}')
                if isinstance(raw_info, (bytes, bytearray)):
                    raw_info = raw_info.decode('utf-8', errors='ignore')

                info = {}
                if isinstance(raw_info, str):
                    try:
                        info = json.loads(raw_info)
                    except Exception:
                        info = {}
                elif isinstance(raw_info, dict):
                    info = raw_info

                for k in ['problem_name', 'task_name', 'task', 'language_instruction']:
                    if info.get(k):
                        task_name = str(info[k])
                        break

                for k in ['bddl_file_name', 'bddl_file', 'bddl_filename', 'bddl']:
                    if info.get(k):
                        bddl_file_name = str(info[k])
                        break

                if bddl_file_name is None:
                    for k in ['bddl_file_name', 'bddl_file', 'bddl_filename', 'bddl']:
                        if k in _hf.attrs:
                            bddl_file_name = str(_hf.attrs[k])
                            break
        except Exception:
            pass

    if not task_name:
        task_name = hdf5_path.stem.replace('_demo', '').replace('_', ' ')

    if not bddl_file_name:
        bddl_file_name = _infer_bddl_from_hdf5_path(hdf5_path)
    elif not str(bddl_file_name).endswith('.bddl'):
        bddl_file_name = f"{bddl_file_name}.bddl"

    return {'task_name': task_name, 'bddl_file_name': bddl_file_name}


def make_libero_env_fn(task_name: str, libero_path=None, image_size: int = 224, bddl_file_name: str = None):
    """Return a callable(seed) -> gym env for a single LIBERO task."""
    def _make(seed: int = 0):
        if not HAS_LIBERO:
            raise RuntimeError(
                'LIBERO simulator not installed. '
                'Re-run the install cell — libero/robomimic/gymnasium must install cleanly.'
            )

        cam_kwargs = {
            'camera_heights': image_size,
            'camera_widths': image_size,
            'camera_names': 'agentview',
        }

        bddl_resolved = _resolve_bddl_path(bddl_file_name, libero_path=libero_path)

        attempt_kwargs = []
        if task_name and bddl_resolved:
            attempt_kwargs.append({'task_name': task_name, 'bddl_file_name': bddl_resolved})
        if bddl_resolved:
            attempt_kwargs.append({'bddl_file_name': bddl_resolved})
        if task_name:
            attempt_kwargs.append({'task_name': task_name})

        env = None
        last_error = None
        for kw in attempt_kwargs:
            try:
                env = OffScreenRenderEnv(**kw, **cam_kwargs)
                break
            except Exception as e:
                last_error = e

        if env is None:
            tried = ', '.join([str(x) for x in attempt_kwargs])
            raise RuntimeError(
                f'Failed to construct OffScreenRenderEnv. Tried: {tried}. Last error: {last_error}'
            )
        if hasattr(env, 'seed'):
            env.seed(seed)
        return env
    return _make


print('Closed-loop helper status: BDDL fallback+resolution is integrated (no patch cell needed).')

In [ ]:
def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)


def schedule_tf_and_free_weight(epoch, num_epochs, cfg):
    p = epoch / max(1, num_epochs - 1)
    tf = max(cfg.tf_ratio_end, cfg.tf_ratio_start - (cfg.tf_ratio_start - cfg.tf_ratio_end) * p)
    free_w = min(cfg.free_w_end, cfg.free_w_start + (cfg.free_w_end - cfg.free_w_start) * p)
    tf_w = 1.0 - free_w
    return tf, tf_w, free_w


def run_epoch(model, loader, opt=None, epoch=0, num_epochs=1, liquid_two_branch=False):
    train = opt is not None
    model.train(train)

    total_loss = 0.0
    total_nll = 0.0
    total_mse = 0.0

    tf_ratio, tf_w, free_w = schedule_tf_and_free_weight(epoch, num_epochs, cfg)

    iterator = tqdm(loader, leave=False) if train else loader
    for batch in iterator:
        batch = to_device(batch, device)

        if liquid_two_branch:
            out_tf = model(batch, action_target=batch['action'], teacher_forcing_ratio=tf_ratio, return_mdn=True)
            out_fr = model(batch, action_target=None, teacher_forcing_ratio=0.0, return_mdn=True)
            nll_tf = mdn_nll_loss(out_tf['aux']['logits'], out_tf['aux']['mu'], out_tf['aux']['log_sigma'], batch['action'])
            nll_fr = mdn_nll_loss(out_fr['aux']['logits'], out_fr['aux']['mu'], out_fr['aux']['log_sigma'], batch['action'])
            loss = tf_w * nll_tf + free_w * nll_fr
            nll = loss
            mse = F.mse_loss(out_fr['pred_actions'], batch['action'])
        else:
            out = model(batch, action_target=batch['action'], teacher_forcing_ratio=0.0, return_mdn=True)
            loss_dict = model.loss(out, batch)
            loss = loss_dict['loss']
            nll = loss_dict['nll']
            mse = loss_dict['mse']

        if train:
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        total_loss += float(loss.item())
        total_nll += float(nll.item())
        total_mse += float(mse.item())

        if train and hasattr(iterator, 'set_postfix'):
            iterator.set_postfix(loss=f'{loss.item():.4f}', nll=f'{nll.item():.4f}')

    n = max(1, len(loader))
    return {
        'loss': total_loss / n,
        'nll': total_nll / n,
        'mse': total_mse / n,
        'teacher_forcing_ratio': tf_ratio,
        'free_running_weight': free_w,
    }


@torch.no_grad()
def evaluate_best_of_k(model, loader, k_values):
    model.eval()
    out = {str(k): {'best_of_k_mse': 0.0, 'sample_mean_mse': 0.0, 'diversity_l2': 0.0, 'smoothness': 0.0} for k in k_values}

    for batch in loader:
        batch = to_device(batch, device)
        pred = model(batch, action_target=None, teacher_forcing_ratio=0.0, return_mdn=True)

        logits = pred['aux']['logits']
        mu = pred['aux']['mu']
        log_sigma = pred['aux']['log_sigma']
        samples = sample_mdn_actions(logits, mu, log_sigma, K=max(k_values))

        for k in k_values:
            m = compute_sample_metrics(samples[:, :k], batch['action'])
            bucket = out[str(k)]
            bucket['best_of_k_mse'] += m['best_of_k_mse']
            bucket['sample_mean_mse'] += m['sample_mean_mse']
            bucket['diversity_l2'] += m['diversity_l2']
            bucket['smoothness'] += m['smoothness']

    n = max(1, len(loader))
    for k in k_values:
        for key in out[str(k)].keys():
            out[str(k)][key] /= n
    return out


@torch.no_grad()
def latency_ms(model, batch, steps=100):
    import time
    batch = to_device(batch, device)
    model.eval()
    for _ in range(20):
        _ = model(batch, action_target=None, teacher_forcing_ratio=0.0, return_mdn=True)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(steps):
        _ = model(batch, action_target=None, teacher_forcing_ratio=0.0, return_mdn=True)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    return 1000.0 * (time.perf_counter() - t0) / steps

In [13]:
# Config + TEST status summary
_test = globals().get('TEST', False)
print(f'TEST             = {_test}')
print(f'epochs           = {cfg.epochs}')
print(f'n_train / n_val  = {cfg.n_train} / {cfg.n_val}')
print(f'pretrained_enc   = {cfg.use_pretrained_encoders}  ({cfg.pretrained_name})')
print(f'freeze_encoders  = {cfg.freeze_encoders}')
print(f'enable_wandb     = {cfg.enable_wandb}')
print(f'resume           = {cfg.resume}')
print(f'save_every       = {cfg.save_every_epochs} epoch(s)')
print(f'run_name         = {cfg.run_name}')
print(f'libero_suite     = {cfg.libero_suite}  (n_tasks={cfg.libero_n_tasks})')
print(f'libero_available = {globals().get("libero_available", "unknown (download cell not run)")}')
print(f'clip_tokenizer   = {"loaded" if globals().get("clip_tokenizer") else "fallback hash"}')
print(f'checkpoint dir   = {globals().get("ckpt_path", "not set (setup cell not run)")}')
print(f'── closed-loop eval ──────────────────────────────────────')
print(f'HAS_LIBERO       = {HAS_LIBERO}')
print(f'n_eval_episodes  = {cfg.n_eval_episodes}  (TEST overrides to 1)')
print(f'eval_horizon     = {cfg.eval_horizon}  steps / episode')
print(f'eval_k_samples   = {cfg.eval_k_samples}  (best-of-K for action selection)')


Smoke test mode: False
Current config: {'use_pretrained_encoders': True, 'epochs': 3}


In [ ]:
import json
from copy import deepcopy
from datetime import datetime, timezone

loader_num_workers = int(getattr(cfg, 'loader_num_workers', 0))
use_pin_memory = bool(torch.cuda.is_available())

# Pull globals set by earlier cells
_clip_tok = globals().get('clip_tokenizer', None)
_lib_root = globals().get('LIBERO_LOCAL_PATH', None)
_lib_path = (Path(str(_lib_root)) / cfg.libero_suite) if _lib_root else None

# Notebook + HDF5 safety: worker processes can deadlock at 0% in some runtimes
if loader_num_workers > 0 and _lib_path is not None and _lib_path.exists() and any(_lib_path.glob('*.hdf5')):
    print('[WARN] Detected HDF5 dataset with DataLoader workers > 0 in notebook runtime.')
    print('[WARN] This can stall at 0% due to multiprocessing/fork deadlocks.')
    print(f'[WARN] Forcing cfg.loader_num_workers: {loader_num_workers} -> 0 for stability.')
    loader_num_workers = 0
    cfg.loader_num_workers = 0

# Backward-compatible path fallback if setup cell was not run
if 'latest_ckpt' not in globals():
    _fallback = Path.cwd() / 'artifacts' / 'notebook_runs' / cfg.run_name / 'checkpoints'
    _fallback.mkdir(parents=True, exist_ok=True)
    latest_ckpt = _fallback / 'latest.pt'
    best_liq_ckpt = _fallback / 'best_liquid.pt'
    best_pi0_ckpt = _fallback / 'best_pi0.pt'

if 'history_json' not in globals() or 'results_json' not in globals():
    _art_dir = latest_ckpt.parent.parent / 'artifacts'
    _art_dir.mkdir(parents=True, exist_ok=True)
    history_json = _art_dir / 'history.json'
    results_json = _art_dir / 'results.json'

resume_flag = bool(getattr(cfg, 'resume', False))
save_every_epochs = int(getattr(cfg, 'save_every_epochs', 1))
run_name = getattr(cfg, 'run_name', 'liquid-pi0-debug')
wandb_run = globals().get('wandb_run', None)

# ---- Data loaders ----
train_dl = DataLoader(
    LiberoAdapterDataset(cfg, split='train', libero_path=_lib_path, clip_tokenizer=_clip_tok),
    batch_size=int(getattr(cfg, 'batch_size', 16)),
    shuffle=True,
    collate_fn=collate,
    num_workers=loader_num_workers,
    pin_memory=use_pin_memory,
 )
val_dl = DataLoader(
    LiberoAdapterDataset(cfg, split='val', libero_path=_lib_path, clip_tokenizer=_clip_tok),
    batch_size=int(getattr(cfg, 'batch_size', 16)),
    shuffle=False,
    collate_fn=collate,
    num_workers=loader_num_workers,
    pin_memory=use_pin_memory,
 )

# ---- Models: shared pretrained encoders + per-model backbone widths ----
liquid_cfg = deepcopy(cfg)
pi0_cfg = deepcopy(cfg)

if bool(getattr(cfg, 'enforce_liquid_half_size', True)):
    liquid_cfg.d_model = int(getattr(cfg, 'liquid_d_model', liquid_cfg.d_model))
    liquid_cfg.n_heads = int(getattr(cfg, 'liquid_n_heads', liquid_cfg.n_heads))
    liquid_cfg.n_layers = int(getattr(cfg, 'liquid_n_layers', liquid_cfg.n_layers))
    liquid_cfg.liquid_hidden = int(getattr(cfg, 'liquid_hidden', liquid_cfg.liquid_hidden))

# Make sure liquid attention heads divide d_model
liquid_cfg.n_heads = max(1, min(liquid_cfg.n_heads, liquid_cfg.d_model))
while liquid_cfg.n_heads > 1 and (liquid_cfg.d_model % liquid_cfg.n_heads != 0):
    liquid_cfg.n_heads -= 1

shared_encoders = PretrainedClipEncoders(cfg)
backbone_liquid = SharedBackbone(liquid_cfg, shared_encoders=shared_encoders)
backbone_pi0 = SharedBackbone(pi0_cfg, shared_encoders=shared_encoders)

liquid = LiquidPolicy(backbone_liquid, liquid_cfg).to(device)
pi0 = Pi0StylePolicy(backbone_pi0, pi0_cfg).to(device)

opt_l = torch.optim.AdamW([p for p in liquid.parameters() if p.requires_grad], lr=cfg.lr, weight_decay=cfg.wd)
opt_p = torch.optim.AdamW([p for p in pi0.parameters() if p.requires_grad], lr=cfg.lr, weight_decay=cfg.wd)

print('Pretrained encoders enabled:', cfg.use_pretrained_encoders and HAS_TRANSFORMERS)
print('Freeze encoders:', cfg.freeze_encoders)
print('num_workers:', loader_num_workers)
print(f'Liquid arch: d_model={liquid_cfg.d_model}, n_layers={liquid_cfg.n_layers}, n_heads={liquid_cfg.n_heads}, hidden={liquid_cfg.liquid_hidden}')
print(f'Pi0 arch   : d_model={pi0_cfg.d_model}, n_layers={pi0_cfg.n_layers}, n_heads={pi0_cfg.n_heads}, hidden={pi0_cfg.pi0_hidden}')

# ---- Parameter counts (exclude shared encoder so ratio reflects model-specific capacity) ----
_enc_trainable = sum(p.numel() for p in shared_encoders.parameters() if p.requires_grad)
_liq_total = count_params(liquid)
_pi0_total = count_params(pi0)
_liq_excl = _liq_total - _enc_trainable
_pi0_excl = _pi0_total - _enc_trainable
print(f'Shared encoder trainable params: {_enc_trainable:,}')
print(f'Liquid params: {_liq_total:,}  (model-specific: {_liq_excl:,})')
print(f'Pi0-style params: {_pi0_total:,}  (model-specific: {_pi0_excl:,})')
_size_ratio = _liq_excl / max(1, _pi0_excl)
print(f'Liquid / Pi0 ratio (model-specific): {_size_ratio:.3f}x')
_target_ratio = float(getattr(cfg, 'target_liquid_to_pi0_ratio', 0.5))
if abs(_size_ratio - _target_ratio) > 0.1:
    print(f'[WARN] Liquid/Pi0 ratio ({_size_ratio:.3f}) differs from target ({_target_ratio:.3f}) by >10%.')
    print('[WARN] Adjust liquid_d_model / liquid_n_layers / liquid_hidden to match target ratio.')

print(f'Train batches: {len(train_dl)} | Val batches: {len(val_dl)}')
print(f'Train samples: {len(train_dl.dataset)} | Val samples: {len(val_dl.dataset)}')

_min_train_batches = int(getattr(cfg, 'min_train_batches', 1))
_strict_min_guard = bool(getattr(cfg, 'strict_min_train_batches', False))
if len(train_dl) < _min_train_batches:
    _msg = (
        f'Train DataLoader too small: len(train_dl)={len(train_dl)} < required {_min_train_batches}. '
        'Increase real LIBERO data coverage (suite/tasks) or reduce batch_size only for smoke tests.'
    )
    if _strict_min_guard:
        raise RuntimeError(_msg + ' Set cfg.strict_min_train_batches=False to auto-relax for this run.')
    print(f'[WARN] {_msg}')
    print(f'[WARN] Auto-relaxing cfg.min_train_batches from {_min_train_batches} to {len(train_dl)} for this run.')
    cfg.min_train_batches = max(1, len(train_dl))

history = {'liquid': [], 'pi0': []}
start_epoch = 0
best_liquid_val = float('inf')
best_pi0_val = float('inf')

# ---- Resume from latest checkpoint ----
if resume_flag and latest_ckpt.exists():
    print(f'Loading checkpoint: {latest_ckpt}')
    ckpt = torch.load(latest_ckpt, map_location=device)
    liquid.load_state_dict(ckpt['liquid_model'])
    pi0.load_state_dict(ckpt['pi0_model'])
    opt_l.load_state_dict(ckpt['liquid_optimizer'])
    opt_p.load_state_dict(ckpt['pi0_optimizer'])
    history = ckpt.get('history', history)
    start_epoch = int(ckpt.get('epoch', -1)) + 1
    best_liquid_val = float(ckpt.get('best_liquid_val', best_liquid_val))
    best_pi0_val = float(ckpt.get('best_pi0_val', best_pi0_val))
    print(f'Resumed from epoch {start_epoch}')
else:
    print('No resume checkpoint found; starting fresh.')

# ---- Training loop ----
for e in range(start_epoch, cfg.epochs):
    liq_train = run_epoch(liquid, train_dl, opt_l, epoch=e, num_epochs=cfg.epochs, liquid_two_branch=True)
    liq_val = run_epoch(liquid, val_dl, None, epoch=e, num_epochs=cfg.epochs, liquid_two_branch=False)

    pi0_train = run_epoch(pi0, train_dl, opt_p, epoch=e, num_epochs=cfg.epochs, liquid_two_branch=False)
    pi0_val = run_epoch(pi0, val_dl, None, epoch=e, num_epochs=cfg.epochs, liquid_two_branch=False)

    history['liquid'].append({'train': liq_train, 'val': liq_val})
    history['pi0'].append({'train': pi0_train, 'val': pi0_val})

    print(
        f"Epoch {e+1:02d}/{cfg.epochs} | "
        f"Liquid NLL train/val: {liq_train['nll']:.4f}/{liq_val['nll']:.4f} | "
        f"Pi0 NLL train/val: {pi0_train['nll']:.4f}/{pi0_val['nll']:.4f} | "
        f"TF={liq_train['teacher_forcing_ratio']:.2f} FW={liq_train['free_running_weight']:.2f}"
    )

    if liq_val['nll'] < best_liquid_val:
        best_liquid_val = liq_val['nll']
        torch.save({'model': liquid.state_dict(), 'epoch': e}, best_liq_ckpt)

    if pi0_val['nll'] < best_pi0_val:
        best_pi0_val = pi0_val['nll']
        torch.save({'model': pi0.state_dict(), 'epoch': e}, best_pi0_ckpt)

    if ((e + 1) % save_every_epochs) == 0:
        torch.save({
            'epoch': e,
            'liquid_model': liquid.state_dict(),
            'pi0_model': pi0.state_dict(),
            'liquid_optimizer': opt_l.state_dict(),
            'pi0_optimizer': opt_p.state_dict(),
            'history': history,
            'best_liquid_val': best_liquid_val,
            'best_pi0_val': best_pi0_val,
            'cfg': cfg.__dict__,
            'timestamp': datetime.now(timezone.utc).isoformat(),
        }, latest_ckpt)

    with open(history_json, 'w') as f:
        json.dump(history, f, indent=2)

    if wandb_run is not None:
        wandb.log({
            'epoch': e + 1,
            'liquid/train_nll': liq_train['nll'],
            'liquid/val_nll': liq_val['nll'],
            'pi0/train_nll': pi0_train['nll'],
            'pi0/val_nll': pi0_val['nll'],
            'liquid/tf_ratio': liq_train['teacher_forcing_ratio'],
            'liquid/free_running_w': liq_train['free_running_weight'],
        })

# ---- Best-of-K evaluation ----
liq_k = evaluate_best_of_k(liquid, val_dl, cfg.k_values)
pi0_k = evaluate_best_of_k(pi0, val_dl, cfg.k_values)

print('\nBest-of-K summary (validation):')
for k in cfg.k_values:
    l = liq_k[str(k)]
    p = pi0_k[str(k)]
    print(
        f"  K={k:>2} | Liquid best={l['best_of_k_mse']:.6f} div={l['diversity_l2']:.4f}"
        f"  |  Pi0 best={p['best_of_k_mse']:.6f} div={p['diversity_l2']:.4f}"
    )

_b = next(iter(val_dl))
liq_latency = latency_ms(liquid, _b)
pi0_latency = latency_ms(pi0, _b)
print(f'\nLatency (ms/batch)  Liquid: {liq_latency:.3f}  Pi0-style: {pi0_latency:.3f}')

# ---- Closed-loop evaluation (requires LIBERO simulator) ----
_closed_loop_results = {}
if HAS_LIBERO:
    print('\n── Closed-loop rollout evaluation ──────────────────────────────────')
    _hdf5_files = sorted(_lib_path.glob('*.hdf5')) if (_lib_path and _lib_path.exists()) else []
    _max_tasks = int(getattr(cfg, 'closed_loop_eval_tasks', len(_hdf5_files) if _hdf5_files else 0))
    _eval_files = _hdf5_files[:max(0, _max_tasks)]

    if _eval_files:
        print(f'  Evaluating {len(_eval_files)} task file(s) with {cfg.n_eval_episodes} episode(s)/task/model')
        _closed_loop_video_dir = artifact_path / 'closed_loop_videos' if 'artifact_path' in globals() else (Path.cwd() / 'artifacts' / 'closed_loop_videos')
        _closed_loop_video_dir.mkdir(parents=True, exist_ok=True)

        for _mname, _model in [('liquid', liquid), ('pi0', pi0)]:
            _task_runs = []
            for _ti, _h5 in enumerate(_eval_files):
                _meta = get_task_metadata_from_hdf5(_h5)
                _eval_task_name = _meta.get('task_name')
                _eval_bddl_file_name = _meta.get('bddl_file_name')
                if not (_eval_task_name or _eval_bddl_file_name):
                    print(f'    [SKIP] Could not parse task metadata from {_h5.name}')
                    continue

                _task_label = _eval_task_name or _eval_bddl_file_name
                _make_env = make_libero_env_fn(
                    _eval_task_name,
                    libero_path=_lib_path,
                    image_size=cfg.image_size,
                    bddl_file_name=_eval_bddl_file_name,
                )
                _tok_fn = make_tokenizer_fn(cfg, clip_tokenizer=_clip_tok, task_name=_task_label)

                print(f'  Evaluating {_mname}: task={_ti+1}/{len(_eval_files)} "{_task_label}"')
                try:
                    _cl = evaluate_closed_loop(
                        _model, _make_env, _tok_fn,
                        episodes=cfg.n_eval_episodes,
                        horizon=cfg.eval_horizon,
                        k_samples=cfg.eval_k_samples,
                        video_dir=_closed_loop_video_dir,
                        video_prefix=f'{_mname}_task{_ti:02d}_closed_loop',
                        video_episode=(0 if _ti == 0 else -1),
                        video_fps=20,
                    )
                    _task_runs.append({'task_label': _task_label, 'hdf5': str(_h5), 'metrics': _cl})
                    print(f'    success_rate = {_cl["success_rate_mean"]:.3f} ± {_cl["success_rate_std"]:.3f}'
                          f' | return = {_cl["return_mean"]:.2f} | steps = {_cl["steps_mean"]:.1f}')
                except Exception as _cl_err:
                    print(f'    [WARN] {_mname} failed on {_h5.name}: {_cl_err}')

            if _task_runs:
                _succ = np.array([t['metrics']['success_rate_mean'] for t in _task_runs], dtype=np.float32)
                _ret = np.array([t['metrics']['return_mean'] for t in _task_runs], dtype=np.float32)
                _steps = np.array([t['metrics']['steps_mean'] for t in _task_runs], dtype=np.float32)
                _agg = {
                    'num_tasks': int(len(_task_runs)),
                    'task_results': _task_runs,
                    'success_rate_mean': float(_succ.mean()),
                    'success_rate_std': float(_succ.std()),
                    'return_mean': float(_ret.mean()),
                    'return_std': float(_ret.std()),
                    'steps_mean': float(_steps.mean()),
                    'steps_std': float(_steps.std()),
                }
                _closed_loop_results[_mname] = _agg
                print(f'  Aggregate {_mname}: success_rate={_agg["success_rate_mean"]:.3f} ± {_agg["success_rate_std"]:.3f}'
                      f' | return={_agg["return_mean"]:.2f} | tasks={_agg["num_tasks"]}')

                if wandb_run is not None:
                    wandb.log({
                        f'closed_loop/{_mname}/success_rate': _agg['success_rate_mean'],
                        f'closed_loop/{_mname}/return': _agg['return_mean'],
                        f'closed_loop/{_mname}/steps': _agg['steps_mean'],
                        f'closed_loop/{_mname}/num_tasks': _agg['num_tasks'],
                    })
    else:
        print('  [SKIP] No LIBERO task HDF5 found — download the data first.')
else:
    print('\n[INFO] LIBERO simulator not installed — closed-loop eval skipped.')
    print('       Re-run the install cell; libero + robomimic + gymnasium must install cleanly.')

# ---- Assemble results ----
results = {
    'run_name': run_name,
    'k_values': cfg.k_values,
    'liquid_k_metrics': liq_k,
    'pi0_k_metrics': pi0_k,
    'latency_ms': {'liquid': liq_latency, 'pi0': pi0_latency},
    'closed_loop': _closed_loop_results,
    'checkpoint_paths': {
        'latest': str(latest_ckpt),
        'best_liquid': str(best_liq_ckpt),
        'best_pi0': str(best_pi0_ckpt),
    },
}
with open(results_json, 'w') as f:
    json.dump(results, f, indent=2)

if wandb_run is not None:
    wandb.log({
        'latency/liquid_ms': liq_latency,
        'latency/pi0_ms': pi0_latency,
        **{f'best_of_k/liquid_k{k}': liq_k[str(k)]['best_of_k_mse'] for k in cfg.k_values},
        **{f'best_of_k/pi0_k{k}': pi0_k[str(k)]['best_of_k_mse'] for k in cfg.k_values},
    })

    art = wandb.Artifact(name=f'{run_name}-artifacts', type='experiment-artifacts')
    for fpath in [history_json, results_json, latest_ckpt, best_liq_ckpt, best_pi0_ckpt]:
        if Path(fpath).exists():
            art.add_file(str(fpath))
    if '_closed_loop_video_dir' in globals() and Path(_closed_loop_video_dir).exists():
        for _vp in sorted(Path(_closed_loop_video_dir).glob('*.mp4')):
            art.add_file(str(_vp))
    wandb_run.log_artifact(art)
    print('W&B artifact uploaded.')

print('\nSaved artifacts:')
print('  history   :', history_json)
print('  results   :', results_json)
print('  latest    :', latest_ckpt)
print('  best liq  :', best_liq_ckpt)
print('  best pi0  :', best_pi0_ckpt)
print('\nTo restart: set cfg.resume=True, rerun setup + this cell.')


## Interface parity checklist

- Same `Batch` keys and tensor shapes
- Same `forward()` contract (`pred_actions`, `aux`)
- Same `loss()` contract (`loss`, `mse`)
- Same train/eval/latency pipeline

This lets you swap only the policy head while keeping all upstream/downstream code fixed.

## MDN, best-of-K, and closed-loop (aligned with training notebook)

This notebook mirrors the core training/evaluation conventions:

1. **MDN outputs** for both heads: `logits`, `mu`, `log_sigma`.
2. **MDN NLL** objective using the same formula as the main training pipeline.
3. **Liquid two-branch training** with teacher-forcing + free-running weighting schedule.
4. **Best-of-K metrics** with matched `K \in \{1,2,5,10\}`.
5. **Trajectory quality metrics** including diversity and smoothness (jerk).
6. **Closed-loop receding-horizon rollout scaffold** for LIBERO envs.

This keeps interfaces shared while matching the shared-backbone comparison protocol semantics.

In [ ]:
# Closed-loop evaluation scaffold aligned with shared-backbone receding-horizon usage

@torch.no_grad()
def policy_first_action(policy, batch, k_samples=10):
    out = policy(batch, action_target=None, teacher_forcing_ratio=0.0, return_mdn=True)
    logits = out['aux']['logits']
    mu = out['aux']['mu']
    log_sigma = out['aux']['log_sigma']

    samples = sample_mdn_actions(logits, mu, log_sigma, K=k_samples)  # [B,K,T,A]
    gt_proxy = mdn_mean_actions(logits, mu)  # [B,T,A]
    errs = ((samples - gt_proxy.unsqueeze(1)) ** 2).mean(dim=(-1, -2))
    best = errs.argmin(dim=1)
    bidx = torch.arange(samples.shape[0], device=samples.device)
    return samples[bidx, best, 0]  # [B,A]


def _extract_video_frame(obs):
    if obs is None:
        return None

    frame = obs.get('agentview_image') if isinstance(obs, dict) else None
    if frame is None and isinstance(obs, dict):
        image_like_keys = [k for k in obs.keys() if 'image' in k.lower() or 'rgb' in k.lower()]
        if image_like_keys:
            frame = obs[image_like_keys[0]]

    if frame is None:
        return None

    frame = np.asarray(frame)
    if frame.ndim != 3:
        return None

    if frame.shape[0] in (1, 3, 4) and frame.shape[-1] not in (1, 3, 4):
        frame = np.transpose(frame, (1, 2, 0))

    if frame.shape[-1] == 1:
        frame = np.repeat(frame, 3, axis=-1)
    elif frame.shape[-1] >= 3:
        frame = frame[..., :3]
    else:
        return None

    if frame.dtype != np.uint8:
        frame = np.clip(frame, 0, 255).astype(np.uint8)

    return frame


def save_rollout_video(frames, out_path, fps=20):
    if not frames:
        raise ValueError('No frames available to save.')

    try:
        import imageio.v2 as imageio
    except Exception:
        import imageio

    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with imageio.get_writer(str(out_path), fps=fps) as writer:
        for frame in frames:
            writer.append_data(frame)
    return out_path


def _safe_env_reset(env):
    out = env.reset()
    if isinstance(out, tuple):
        if len(out) >= 1:
            obs = out[0]
            info = out[1] if len(out) >= 2 and isinstance(out[1], dict) else {}
            return obs, info
    return out, {}


def _safe_env_step(env, action):
    out = env.step(action)
    if isinstance(out, tuple):
        if len(out) == 5:
            obs, reward, terminated, truncated, info = out
            return obs, reward, bool(terminated), bool(truncated), info if isinstance(info, dict) else {}
        if len(out) == 4:
            obs, reward, done, info = out
            return obs, reward, bool(done), False, info if isinstance(info, dict) else {}
    raise RuntimeError(f'Unexpected env.step() return format: {type(out)}')


@torch.no_grad()
def closed_loop_rollout(policy, env, tokenizer_fn, horizon=300, k_samples=10, video_path=None, video_fps=20):
    obs, _ = _safe_env_reset(env)
    total_reward = 0.0
    steps = 0
    done = False
    info = {}
    frames = []

    first_frame = _extract_video_frame(obs)
    if first_frame is not None:
        frames.append(first_frame)

    while not done and steps < horizon:
        batch = tokenizer_fn(obs)
        batch = to_device(batch, device)

        a = policy_first_action(policy, batch, k_samples=k_samples)[0].detach().cpu().numpy()
        obs, reward, terminated, truncated, info = _safe_env_step(env, a)

        frame = _extract_video_frame(obs)
        if frame is not None:
            frames.append(frame)

        total_reward += float(reward)
        done = bool(terminated or truncated)
        steps += 1

    saved_video = None
    if video_path is not None and frames:
        try:
            saved_video = str(save_rollout_video(frames, video_path, fps=video_fps))
        except Exception as e:
            print(f'[WARN] Video save failed: {e}')

    return {
        'return': float(total_reward),
        'steps': int(steps),
        'success': float(info.get('success', 0.0)),
        'video_path': saved_video,
        'num_video_frames': int(len(frames)),
    }


@torch.no_grad()
def evaluate_closed_loop(policy, make_env_fn, tokenizer_fn, episodes=20, horizon=300, k_samples=10, seed=0, video_dir=None, video_prefix='rollout', video_episode=0, video_fps=20):
    returns, successes, steps_list = [], [], []
    saved_videos = []
    for ep in range(episodes):
        env = make_env_fn(seed + ep)
        episode_video_path = None
        if video_dir is not None and ep == int(video_episode):
            episode_video_path = Path(video_dir) / f'{video_prefix}_ep{ep:02d}.mp4'
        out = closed_loop_rollout(
            policy, env, tokenizer_fn, horizon=horizon, k_samples=k_samples,
            video_path=episode_video_path, video_fps=video_fps,
        )
        env.close()
        returns.append(out['return'])
        successes.append(out['success'])
        steps_list.append(out['steps'])
        if out.get('video_path'):
            saved_videos.append(out['video_path'])

    success_mean = float(np.mean(successes)) if successes else 0.0
    success_std = float(np.std(successes)) if successes else 0.0
    return {
        'return_mean': float(np.mean(returns)) if returns else 0.0,
        'return_std': float(np.std(returns)) if returns else 0.0,
        'success_rate': success_mean,
        'success_rate_mean': success_mean,
        'success_rate_std': success_std,
        'steps_mean': float(np.mean(steps_list)) if steps_list else 0.0,
        'steps_std': float(np.std(steps_list)) if steps_list else 0.0,
        'episodes': int(episodes),
        'video_paths': saved_videos,
        'videos': saved_videos,
    }

Closed-loop scaffold ready. Plug in LIBERO env + tokenizer_fn to run rollouts.


In [ ]:
# Progress patch for closed-loop evaluation (episode-level progress + timing)
from tqdm.auto import tqdm
import time

@torch.no_grad()
def evaluate_closed_loop(policy, make_env_fn, tokenizer_fn, episodes=20, horizon=300, k_samples=10, seed=0, video_dir=None, video_prefix='rollout', video_episode=0, video_fps=20, show_progress=True):
    returns, successes, steps_list = [], [], []
    saved_videos = []

    ep_iter = range(episodes)
    if show_progress:
        ep_iter = tqdm(ep_iter, desc=f'{video_prefix}: episodes', leave=False)

    t_start_all = time.perf_counter()
    for ep in ep_iter:
        t_ep = time.perf_counter()
        env = make_env_fn(seed + ep)
        episode_video_path = None
        if video_dir is not None and ep == int(video_episode):
            episode_video_path = Path(video_dir) / f'{video_prefix}_ep{ep:02d}.mp4'

        out = closed_loop_rollout(
            policy, env, tokenizer_fn, horizon=horizon, k_samples=k_samples,
            video_path=episode_video_path, video_fps=video_fps,
        )
        env.close()

        returns.append(out['return'])
        successes.append(out['success'])
        steps_list.append(out['steps'])
        if out.get('video_path'):
            saved_videos.append(out['video_path'])

        ep_sec = time.perf_counter() - t_ep
        if show_progress and hasattr(ep_iter, 'set_postfix'):
            ep_iter.set_postfix(
                ep=f'{ep+1}/{episodes}',
                steps=int(out['steps']),
                succ=f"{float(out['success']):.2f}",
                sec=f"{ep_sec:.1f}"
            )
        elif (ep + 1) % max(1, episodes // 5) == 0:
            print(f'  progress: {ep+1}/{episodes} episodes | last_ep={ep_sec:.1f}s | steps={int(out["steps"])}')

    total_sec = time.perf_counter() - t_start_all
    success_mean = float(np.mean(successes)) if successes else 0.0
    success_std = float(np.std(successes)) if successes else 0.0
    print(
        f'Closed-loop done: episodes={episodes}, horizon={horizon}, '
        f'total_time={total_sec/60.0:.1f} min, avg_ep={total_sec/max(1, episodes):.1f}s'
    )

    return {
        'return_mean': float(np.mean(returns)) if returns else 0.0,
        'return_std': float(np.std(returns)) if returns else 0.0,
        'success_rate': success_mean,
        'success_rate_mean': success_mean,
        'success_rate_std': success_std,
        'steps_mean': float(np.mean(steps_list)) if steps_list else 0.0,
        'steps_std': float(np.std(steps_list)) if steps_list else 0.0,
        'episodes': int(episodes),
        'video_paths': saved_videos,
        'videos': saved_videos,
        'total_time_sec': float(total_sec),
    }

print('Progress patch active: evaluate_closed_loop now shows per-episode progress/timing.')

In [ ]:
# Deprecated compatibility cell
# BDDL fallback and resolution are now built into the consolidated helper cell above.
# No manual patch bootstrap step is required.
print('No-op: closed-loop patch bootstrap has been consolidated upstream.')

In [ ]:
# Deprecated duplicate rerun cell
# Use the final "Closed-loop only rerun" cell below (single canonical rerun path).
print('No-op: duplicate rerun cell disabled to avoid running closed-loop eval twice.')

In [ ]:
# Deprecated compatibility cell
# Robust BDDL metadata/path handling now lives in the consolidated helper cell above.
# Keep this no-op for backward-compatible run orders.
print('No-op: patch logic has been consolidated into the main helper path.')

In [ ]:
# Closed-loop only rerun (after training cell has executed once)
import time

_hdf5_files = sorted(_lib_path.glob('*.hdf5')) if (_lib_path and _lib_path.exists()) else []
assert _hdf5_files, "No HDF5 files found. Re-run dataset download cell."

_meta = get_task_metadata_from_hdf5(_hdf5_files[0])
_eval_task_name = _meta.get('task_name')
_eval_bddl_file_name = _meta.get('bddl_file_name')
_eval_bddl_resolved = _resolve_bddl_path(_eval_bddl_file_name, libero_path=_lib_path)
_task_label = _eval_task_name or _eval_bddl_file_name

print('HDF5 file:', _hdf5_files[0])
print('task_name:', _eval_task_name)
print('bddl (raw):', _eval_bddl_file_name)
print('bddl (resolved):', _eval_bddl_resolved)
print(f'Planned eval: 2 models × {cfg.n_eval_episodes} episodes/model × horizon {cfg.eval_horizon} steps')
print('Tip: first episode can be slower due to simulator/assets initialization.')

_make_env = make_libero_env_fn(
    _eval_task_name,
    libero_path=_lib_path,
    image_size=cfg.image_size,
    bddl_file_name=_eval_bddl_file_name,

)
_tok_fn = make_tokenizer_fn(cfg, clip_tokenizer=_clip_tok, task_name=_task_label)

_video_dir = artifact_path / 'closed_loop_videos' if 'artifact_path' in globals() else (Path.cwd() / 'artifacts' / 'closed_loop_videos')
_video_dir.mkdir(parents=True, exist_ok=True)

_all_start = time.perf_counter()
for name, model, ckpt in [("liquid", liquid, best_liq_ckpt), ("pi0", pi0, best_pi0_ckpt)]:
    _model_start = time.perf_counter()

    if ckpt.exists():
        payload = torch.load(ckpt, map_location=device)
        model.load_state_dict(payload['model'])
        print(f'Loaded best checkpoint for {name}: {ckpt}')
    else:
        print(f'[WARN] Best checkpoint missing for {name}; using in-memory weights.')

    out = evaluate_closed_loop(
        model, _make_env, _tok_fn,
        episodes=cfg.n_eval_episodes,
        horizon=cfg.eval_horizon,
        k_samples=cfg.eval_k_samples,
        video_dir=_video_dir,
        video_prefix=f'{name}_closed_loop',
        video_episode=0,
        video_fps=20,
        show_progress=True,
    )

    _model_sec = time.perf_counter() - _model_start
    print(name, out)
    print(f'  elapsed for {name}: {_model_sec/60.0:.1f} min')

    if out.get('video_paths'):
        print(f"  saved video: {out['video_paths'][0]}")

        if 'wandb_run' in globals() and (wandb_run is not None):
            try:
                wandb.log({
                    f'closed_loop/{name}/video': wandb.Video(out['video_paths'][0], fps=20, format='mp4'),
                    f'closed_loop/{name}/success_rate': out.get('success_rate_mean', 0.0),
                    f'closed_loop/{name}/return': out.get('return_mean', 0.0),
                    f'closed_loop/{name}/steps': out.get('steps_mean', 0.0),
                })
                print(f"  logged to W&B: {out['video_paths'][0]}")
            except Exception as _wb_err:
                print(f"  [WARN] W&B video log failed for {name}: {_wb_err}")

if 'wandb_run' in globals() and (wandb_run is not None):
    try:
        _all_videos = sorted(_video_dir.glob('*.mp4'))
        if _all_videos:
            _video_art = wandb.Artifact(name=f'{cfg.run_name}-closed-loop-videos', type='rollout-videos')
            for _vp in _all_videos:
                _video_art.add_file(str(_vp))
            wandb_run.log_artifact(_video_art)
            print(f'Uploaded {len(_all_videos)} rollout video(s) to W&B artifact.')
    except Exception as _wb_art_err:
        print(f'[WARN] W&B video artifact upload failed: {_wb_art_err}')

print(f'Total rerun elapsed: {(time.perf_counter() - _all_start)/60.0:.1f} min')

In [ ]:
# Final paper-ready closed-loop summary table
from pathlib import Path
import json

# Resolve results payload from memory or disk
_payload = globals().get('results', None)
if _payload is None:
    _results_path = globals().get('results_json', Path.cwd() / 'artifacts' / 'results.json')
    _results_path = Path(_results_path)
    if not _results_path.exists():
        raise FileNotFoundError(f"No in-memory results and file not found: {_results_path}")
    with _results_path.open('r') as f:
        _payload = json.load(f)

_closed = _payload.get('closed_loop', {})

print("\nClosed-loop LIBERO summary (aggregated across evaluated tasks)")
print(f"{'Model':<10} {'Success Mean':>14} {'Success Std':>12} {'Num Tasks':>10} {'Return Mean':>12}")
print("-" * 66)
for _name in ('liquid', 'pi0'):
    _m = _closed.get(_name, {})
    print(
        f"{_name:<10} "
        f"{float(_m.get('success_rate_mean', 0.0)):>14.4f} "
        f"{float(_m.get('success_rate_std', 0.0)):>12.4f} "
        f"{int(_m.get('num_tasks', 0)):>10d} "
        f"{float(_m.get('return_mean', 0.0)):>12.4f}"
    )

# Optional per-task preview (first 5 tasks)
for _name in ('liquid', 'pi0'):
    _tasks = (_closed.get(_name, {}) or {}).get('task_results', [])
    if not _tasks:
        continue
    print(f"\n{_name} task-level preview (first {min(5, len(_tasks))}):")
    print(f"{'Task':<42} {'Success':>8} {'Return':>10}")
    for _t in _tasks[:5]:
        _label = str(_t.get('task_label', 'unknown'))[:42]
        _metrics = _t.get('metrics', {})
        print(f"{_label:<42} {float(_metrics.get('success_rate_mean', 0.0)):>8.3f} {float(_metrics.get('return_mean', 0.0)):>10.3f}")